# Cahoy 2010 vs Aurora PICASO comparison

Run after any two-stage Cahoy replication spectra are available. By default this notebook compares only the NetCDF files that already exist, so it works before all `304` spectra finish.

```bash
source env/activate_aurora_picaso4_job.sh
bash roadrunner_egp/aurora_subneptune_grid/scripts/install_cahoy2010_reference.sh
```

In [12]:
import json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in [cwd, *cwd.parents]
     if (p / "roadrunner_egp" / "aurora_subneptune_grid" / "src" / "aurora_grid").exists()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not find repo root containing roadrunner_egp/aurora_subneptune_grid/src/aurora_grid")

GRID_ROOT = REPO_ROOT / "roadrunner_egp" / "aurora_subneptune_grid"
MODEL = "aurora_cahoy2010_replication_v0"
MANIFEST = GRID_ROOT / "manifests" / f"{MODEL}_manifest.csv"
NC_ROOT = GRID_ROOT / "outputs" / MODEL / "nc"
COMPARE_ROOT = GRID_ROOT / "outputs" / MODEL / "cahoy_compare"
PLOT_ROOT = COMPARE_ROOT / "plots"

for extra in (GRID_ROOT / "src", REPO_ROOT / "roadrunner_egp"):
    s = str(extra)
    if s not in sys.path:
        sys.path.insert(0, s)

from aurora_grid.cahoy_compare import (
    compare_manifest_outputs,
    compare_nc_to_cahoy,
    ensure_reference_installed,
    metrics_to_records,
)

REFERENCE_ROOT = ensure_reference_installed()
COMPARE_ROOT.mkdir(parents=True, exist_ok=True)
PLOT_ROOT.mkdir(parents=True, exist_ok=True)

# Keep this True while the HPC array is still generating spectra.
AVAILABLE_ONLY = True
N_MANIFEST_ROWS = len(pd.read_csv(MANIFEST, usecols=["run_index"]))
N_EXISTING_NC = len(list(NC_ROOT.glob("run_*.nc")))

print("REPO_ROOT:", REPO_ROOT)
print("GRID_ROOT:", GRID_ROOT)
print("Reference:", REFERENCE_ROOT)
print("NC dir:", NC_ROOT)
print(f"existing NetCDF files: {N_EXISTING_NC} / {N_MANIFEST_ROWS}")

REPO_ROOT: /home/u11/danielxinhuang/Documents/aurora
GRID_ROOT: /home/u11/danielxinhuang/Documents/aurora/roadrunner_egp/aurora_subneptune_grid
Reference: /home/u11/danielxinhuang/Documents/aurora/roadrunner_egp/aurora_subneptune_grid/data/cahoy2010_reference/Cahoy_et_al_2010_Albedo_Spectra/albedo_spectra
NC dir: /home/u11/danielxinhuang/Documents/aurora/roadrunner_egp/aurora_subneptune_grid/outputs/aurora_cahoy2010_replication_v0/nc
existing NetCDF files: 150 / 304


In [13]:
results = compare_manifest_outputs(
    MANIFEST,
    NC_ROOT,
    reference_root=REFERENCE_ROOT,
    existing_only=AVAILABLE_ONLY,
)
records = metrics_to_records(results)
df = pd.DataFrame(records)
if df.empty:
    df_ok = df.copy()
    print(f"No NetCDF files found in {NC_ROOT}")
else:
    df_ok = df[df["status"] == "ok"].copy()
    print(f"compared: {len(df_ok)} available files / {N_MANIFEST_ROWS} manifest rows")
    print(df["status"].value_counts(dropna=False).to_string())
if not df_ok.empty:
    display(df_ok.sort_values("rmse", ascending=False).head(10))
elif not df.empty:
    display(df.head())

TypeError: compare_manifest_outputs() got an unexpected keyword argument 'existing_only'

In [ ]:
metrics_path = COMPARE_ROOT / "cahoy_compare_metrics.csv"
if not df.empty:
    df.to_csv(metrics_path, index=False)
status_counts = df["status"].value_counts().to_dict() if "status" in df else {}
summary = {
    "manifest": str(MANIFEST),
    "nc_root": str(NC_ROOT),
    "available_only": AVAILABLE_ONLY,
    "n_manifest_rows": int(N_MANIFEST_ROWS),
    "n_existing_nc": int(N_EXISTING_NC),
    "n_rows_checked": int(len(df)),
    "n_compared": int(status_counts.get("ok", 0)),
    "n_missing_nc": int(status_counts.get("missing_nc", 0)),
    "n_failed": int(sum(count for status, count in status_counts.items() if status not in ["ok", "missing_nc"])),
    "median_rmse": float(df_ok["rmse"].median()) if not df_ok.empty else None,
    "max_rmse": float(df_ok["rmse"].max()) if not df_ok.empty else None,
}
if not df_ok.empty:
    summary["worst_case"] = str(df_ok.sort_values("rmse", ascending=False).iloc[0]["cahoy_reference_name"])
(COMPARE_ROOT / "cahoy_compare_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
if not df.empty:
    print(f"wrote {metrics_path}")

In [ ]:
def plot_pair(metrics, arrays, out_path=None):
    wave = arrays["wavelength_um"]
    fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True, gridspec_kw={"height_ratios": [3, 1]})
    axes[0].plot(wave, arrays["cahoy_albedo"], label="Cahoy 2010", color="0.25")
    axes[0].plot(wave, arrays["aurora_albedo"], label="Aurora PICASO", color="C0")
    axes[0].set_ylabel("Albedo")
    axes[0].set_title(metrics.cahoy_reference_name)
    axes[0].legend()
    axes[0].grid(alpha=0.25)
    axes[1].plot(wave, arrays["residual"], color="C3")
    axes[1].axhline(0, color="0.4", lw=0.8)
    axes[1].set_xlabel("Wavelength (um)")
    axes[1].set_ylabel("Aurora - Cahoy")
    axes[1].grid(alpha=0.25)
    fig.suptitle(f"RMSE={metrics.rmse:.4f}  phase={metrics.phase_deg:.0f} deg", fontsize=10)
    fig.tight_layout()
    if out_path is not None:
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
    return fig


ok = [item for item in results if item[2] is None]
for metrics, arrays, _ in sorted(ok, key=lambda item: item[0].rmse, reverse=True)[:6]:
    stem = metrics.cahoy_reference_name.replace(".dat", "")
    plot_pair(metrics, arrays, PLOT_ROOT / f"{stem}.png")
    plt.show()
print(f"wrote {min(len(ok), 6)} plots to {PLOT_ROOT}")

In [ ]:
# Pick one case to inspect interactively. If CASE_NAME is not available yet,
# show the worst currently compared case instead.
CASE_NAME = "Jupiter_1x_0.8AU_0deg.dat"
case = next((item for item in ok if item[0].cahoy_reference_name == CASE_NAME), None)
if case is None and ok:
    case = max(ok, key=lambda item: item[0].rmse)
    print(f"{CASE_NAME} is not available yet; showing {case[0].cahoy_reference_name} instead.")

if case is not None:
    metrics, arrays, _ = case
    plot_pair(metrics, arrays)
    plt.show()
else:
    print(f"No comparable NetCDF files found in {NC_ROOT}")